# 2.4 為什麼需要非線性？

# 第 2 章：讓模型看更長的上下文

前置：第1章的索引、loss、梯度。這章只增加固定上下文，沒有 attention。向量是可以學習的數字列表，不是先驗語意標籤。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：上下文像把幾張卡片一起攤開**

拼接順序保留位置；窗口固定長度。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/context.svg")))

**先想一想：**兩層 Linear 一定比一層有更多種函數嗎？

純 Linear 疊再多層，仍可合成一個 Linear。加入彎曲的 activation 才能表達輸入區域不同的規則。

**把直覺寫成數學：**$A(Bx+b)+a=(AB)x+(Ab+a)$；ReLU(x)=max(0,x) 無法整體寫成線性。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
a, b = nn.Linear(3, 4), nn.Linear(4, 2)
x = torch.randn(5, 3)
combined = x @ (b.weight @ a.weight).T + (b.weight @ a.bias + b.bias)
print("線性合成誤差", (b(a(x)) - combined).abs().max().item())
print("加入ReLU的改變", (b(F.relu(a(x))) - combined).abs().mean().item())

**如何讀結果：**非線性改變可表示的函數家族；增加參數不等於一定有更好的泛化。

**只改一件事：**只把 ReLU 改成 tanh。
